In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.recommender.baseline import popular_products, recommend_popular, user_frequent_products
from src.recommender.evaluation import f1_at_k, precision_at_k
from src.recommender.knn import ItemKNN

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": .3,
                     "axes.spines.top": False, "axes.spines.right": False})
COLORS = {"Baseline": "#8c8c8c", "Repeat": "#dd8452", "KNN": "#4c72b0"}
MODELS = list(COLORS)
K_MAX = 10
N_USERS = 5000

In [ ]:
train = pd.read_csv(ROOT / "data/processed/train.csv")
test = pd.read_csv(ROOT / "data/processed/test.csv")

actual = test.groupby("user_id")["product_id"].apply(set).to_dict()
users = sorted(set(actual) & set(train["user_id"].unique()))
users = sorted(pd.Series(users).sample(min(N_USERS, len(users)), random_state=0))

popular = popular_products(train)
frequent = user_frequent_products(train[train["user_id"].isin(set(users))])
knn = ItemKNN().fit(train, history_users=users)

recs = {m: {} for m in MODELS}
for u in users:
    recs["Baseline"][u] = recommend_popular(popular, k=K_MAX)
    recs["Repeat"][u] = ([int(p) for p in frequent.get(u, [])] + recommend_popular(popular, k=K_MAX))[:K_MAX]
    recs["KNN"][u] = knn.recommend(u, k=K_MAX)

print("Users evaluated:", len(users))

In [ ]:
def per_user_metrics(model, k):
    rows = []
    for u in users:
        r, a = recs[model][u][:k], actual[u]
        hits = len(set(r) & a)
        rows.append({"user": u, "model": model, "k": k, "hits": hits,
                     "precision": precision_at_k(r, a, k), "recall": hits / len(a),
                     "hit": float(hits > 0), "f1": f1_at_k(r, a, k)})
    return pd.DataFrame(rows)

metrics = pd.concat([per_user_metrics(m, k) for m in MODELS for k in range(1, K_MAX + 1)], ignore_index=True)
metrics.groupby(["model", "k"])[["precision", "recall", "hit", "f1"]].mean().unstack(0).round(4).loc[[1, 3, 5, 10]]

In [ ]:
rng = np.random.default_rng(0)

def bootstrap_ci(values, n=2000):
    values = np.asarray(values)
    means = [rng.choice(values, len(values)).mean() for _ in range(n)]
    return values.mean(), *np.percentile(means, [2.5, 97.5])

k = 5
names = {"precision": "Precision@5", "recall": "Recall@5", "hit": "HitRate@5", "f1": "F1@5"}
fig, axes = plt.subplots(1, 4, figsize=(15, 3.8))
for ax, (col, title) in zip(axes, names.items()):
    for i, model in enumerate(MODELS):
        m, lo, hi = bootstrap_ci(metrics[(metrics.model == model) & (metrics.k == k)][col])
        ax.bar(i, m, color=COLORS[model], yerr=[[m - lo], [hi - m]], capsize=5)
        ax.text(i, hi, f"{m:.3f}", ha="center", va="bottom")
    ax.set_xticks(range(len(MODELS))); ax.set_xticklabels(MODELS); ax.set_title(title)
fig.suptitle(f"Metrics at k={k} with 95% bootstrap CI (n={len(users)} users)")
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, (col, title) in zip(axes, [("precision", "Precision@k"), ("recall", "Recall@k"), ("hit", "HitRate@k")]):
    for model in MODELS:
        s = metrics[metrics.model == model].groupby("k")[col].mean()
        ax.plot(s.index, s.values, marker="o", color=COLORS[model], label=model)
    ax.set_title(title); ax.set_xlabel("k"); ax.legend()
plt.tight_layout()

In [ ]:
k = 5
hits = {m: metrics[(metrics.model == m) & (metrics.k == k)].set_index("user")["hits"] for m in MODELS}

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
dist = pd.DataFrame({m: hits[m].clip(upper=4).value_counts() for m in MODELS}).fillna(0).sort_index()
dist.index = [str(i) if i < 4 else "4+" for i in dist.index]
dist.plot.bar(ax=ax[0], color=[COLORS[m] for m in MODELS], rot=0, title="Users by number of hits in top-5")
ax[0].set_xlabel("hits")

diff = hits["KNN"] - hits["Baseline"]
outcome = pd.Series({"KNN better": (diff > 0).sum(), "tie": (diff == 0).sum(), "Baseline better": (diff < 0).sum()})
outcome.plot.bar(ax=ax[1], color=[COLORS["KNN"], "#cccccc", COLORS["Baseline"]], rot=0, title="KNN vs Baseline per user (hits in top-5)")
for i, v in enumerate(outcome):
    ax[1].text(i, v, str(v), ha="center", va="bottom")
plt.tight_layout()
print("Total hits:", {m: int(h.sum()) for m, h in hits.items()})

In [ ]:
hist_len = train[train["user_id"].isin(set(users))].groupby("user_id")["product_id"].nunique()
bucket = pd.cut(hist_len, [0, 10, 25, 50, 100, np.inf], labels=["<=10", "11-25", "26-50", "51-100", "100+"])

m5 = metrics[metrics.k == 5].copy()
m5["bucket"] = m5["user"].map(bucket)
tbl = m5.groupby(["bucket", "model"], observed=True)["precision"].mean().unstack()[MODELS]
cnt = m5[m5.model == "KNN"].groupby("bucket", observed=True).size()

ax = tbl.plot.bar(color=[COLORS[m] for m in MODELS], rot=0, figsize=(9, 3.8), title="Precision@5 by number of distinct products in user's history")
for i, n in enumerate(cnt):
    ax.text(i, 0, f"n={n}", ha="center", va="bottom", fontsize=8)
ax.set_xlabel("distinct products in train history")

In [ ]:
candidates = set(knn.items.tolist())
top5 = set(popular[:5])
rows = []
for u in users:
    a = actual[u]
    rows.append({"products": len(a), "in_knn_candidates": len(a & candidates), "in_popular_top5": len(a & top5)})

history_sets = train[train["user_id"].isin(set(users))].groupby("user_id")["product_id"].apply(set).to_dict()
already = sum(len(actual[u] & history_sets.get(u, set())) for u in users)

cover = pd.DataFrame(rows).sum()
cover["already_bought_by_user"] = already
share = (cover / cover["products"]).drop("products")
ax = share.plot.bar(color="#55a868", rot=0, figsize=(8, 3.8), title="Share of last-order products that could be hit")
for i, name in enumerate(share.index):
    ax.text(i, share[name], f"{int(cover[name])}/{int(cover['products'])}", ha="center", va="bottom")